# Baseline classifier: training, testing, running

Fine-tunes an Ultralytics YOLO classification model (`yolo26n-cls.pt`) on
SID-Set to tell real photos from AI-generated/tampered ones.

This notebook drives the pipeline entirely through the shared classes in
`packages/models/normal_classifier`, instead of ad-hoc inline code:

- **Training + testing** — `NormalClassifierTrainer`, which extends
  `shared_types.TrainableModel` (`.train()` / `.evaluate()` / `.save()` / `.load()`).
- **Running (inference)** — `NormalClassifierDetector`, which implements
  `shared_types.EnsembleDetector` (`.predict()`) — the same "ready" contract
  `apps/web`'s Streamlit demo already knows how to consume.

Data comes from `data.datasets.load_sid_subset()` (streams SID-Set from
Hugging Face) via `data.datasets.to_labeled_samples()`, which adapts it into
the shared `LabeledImageSample` type both classes above are built around.


## 1. Setup — clone the repo, install deps, wire up imports

In [ ]:
# Install uv, then use it to resolve + install a workspace package by name
# alone. `uv sync --package <name>` walks this monorepo's tool.uv.sources
# links automatically (shared_types, image_io, data, ... however many
# levels deep) -- there's no manual "-e path/to/each/dependency" list to
# keep in sync as the dependency graph changes. To work on a different
# package later, just change PACKAGE below.
!git clone --branch dashboard --depth 1 https://github.com/Zhongbob/TikTokTechJam2026.git
%cd TikTokTechJam2026
%pip install -q uv

PACKAGE = "normal_classifier"

import sys

# Pin uv to the interpreter this notebook's kernel is already running, so
# the venv it builds is ABI-compatible with what we import below (rather
# than uv provisioning a separate Python of its own).
!uv sync --package {PACKAGE} --python {sys.executable}

# Make that venv's installed packages importable from this notebook's own
# kernel process (Colab doesn't support pointing a running kernel at a
# different venv, so we add its site-packages to sys.path instead).
sys.path.insert(0, f".venv/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages")


## 2. Load training + validation data from SID-Set

In [ ]:
from data.datasets import load_sid_subset, to_labeled_samples

# SID-Set exposes separate train/validation splits on the Hub; mirror that
# split here rather than carving one out of a single pull.
train_images, train_metadata = load_sid_subset(images_per_label=1000, split="train")
val_images, val_metadata = load_sid_subset(images_per_label=200, split="validation")

train_samples = to_labeled_samples(train_images, train_metadata)
val_samples = to_labeled_samples(val_images, val_metadata)

print(f"train samples: {len(train_samples)}, val samples: {len(val_samples)}")


## 3. Train

`NormalClassifierTrainer.train()` exports `train_samples`/`val_samples` into
the `real/`, `ai_generated/` class-folder layout Ultralytics' classification
trainer expects, then fine-tunes `yolo26n-cls.pt` on them.

In [ ]:
from normal_classifier import NormalClassifierTrainer

trainer = NormalClassifierTrainer(base_weights="yolo26n-cls.pt", image_size=224)
result = trainer.train(
    train_samples,
    val_samples=val_samples,
    output_dir="SID_YOLO",
    epochs=100,
    batch=32,
    patience=10,
    device="cpu",
    plots=True,
)
result


## 4. Test

The "testing" stage — score the trained model against a held-out set via
`.evaluate()`. SID-Set only exposes train/validation splits, so this reuses
`val_samples`; swap in a separate held-out set here if you have one.

In [ ]:
metrics = trainer.evaluate(val_samples, output_dir="SID_YOLO_eval")
print("Held-out evaluation metrics:", metrics)


In [ ]:
trainer.save("normal_classifier.pt")
print("Saved checkpoint to normal_classifier.pt")


## 5. Run (inference)

The "running" stage — `NormalClassifierDetector` wraps the saved checkpoint
and implements the same `EnsembleDetector` contract `apps/web` consumes, so
this class can be dropped straight into
`apps/web/src/web/services/factory.py`'s `get_detector()` once ready.

In [ ]:
from normal_classifier import NormalClassifierDetector

detector = NormalClassifierDetector.from_checkpoint("normal_classifier.pt")

for sample in val_samples[:5]:
    detection = detector.predict(sample.image)
    print(
        f"true={sample.metadata['label_name']:<10} "
        f"predicted={detection.verdict:<12} "
        f"p(ai_generated)={detection.ai_generated_probability:.2f}"
    )
